This is comparing the top 5 known binders based K_i

In [ ]:
import os
import numpy as np
import pandas as pd
from vina import Vina

In [5]:
#input file database as pandas object
input_tsv = "/Users/stuytschaevers/Documents/Yale/spring2026/comp_proteomics/final_project/TNFA_CBB_FINAL_PROJECT.tsv"
df = pd.read_csv(input_tsv, sep="\t")

/var/folders/1x/tcsvk1ss6r7c6d5l0qxjgycr0000gn/T/ipykernel_92136/2659831835.py:3: DtypeWarning: Columns (9,11,15,35,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_tsv, sep="\t")


In [7]:
# Sort the file by "Ki (nM)" column in ascending order
df_sorted = df.sort_values(by="Ki (nM)", ascending=True)

#print top 5 rows of the sorted dataframe
print(df_sorted.head())


   BindingDB Reactant_set_id  \
0                     220373   
1                     220221   
2                     220220   
3                     220264   
4                     220261   

                                       Ligand SMILES  \
0  COc1ccc2CN(C[C@]3(NC(=O)NC3=O)C#Cc3cncc(c3)-c3...   
1  COc1ccc2CN(C[C@]3(NC(=O)NC3=O)C#Cc3cc(F)c4c(c3...   
2  COc1ccc2CN(C[C@]3(NC(=O)NC3=O)C#Cc3cc(F)c4c(c3...   
3  COc1ccc2CN(C[C@]3(NC(=O)NC3=O)C#Cc3ccc(cc3)-c3...   
4  COc1ccc2CN(C[C@]3(NC(=O)NC3=O)C#Cc3ccc(cc3)-c3...   

                                        Ligand InChI  \
0  InChI=1S/C23H18N6O4/c1-33-18-3-2-15-12-29(20(3...   
1  InChI=1S/C22H15F2N5O5/c1-34-14-3-2-11-8-29(19(...   
2  InChI=1S/C22H16FN5O5/c1-33-13-3-2-12-9-28(19(3...   
3  InChI=1S/C30H24N6O5/c1-35-15-21(14-31-35)24-9-...   
4  InChI=1S/C31H23N5O5/c1-41-23-7-6-22-17-36(28(3...   

              Ligand InChI Key  BindingDB MonomerID BindingDB Ligand Name  \
0  GNXUVQQPJKTEIJ-UHFFFAOYSA-N               102777     

In [ ]:
# extract just the SMILES column for docking for the top 5 rows
smiles_list = df_sorted["Ligand SMILES"].head().tolist()
print(smiles_list)

['COc1ccc2CN(C[C@]3(NC(=O)NC3=O)C#Cc3cncc(c3)-c3cn[nH]c3)C(=O)c2c1 |r|', 'COc1ccc2CN(C[C@]3(NC(=O)NC3=O)C#Cc3cc(F)c4c(c3)[nH][nH]c4=O)C(=O)c2c1F |r|', 'COc1ccc2CN(C[C@]3(NC(=O)NC3=O)C#Cc3cc(F)c4c(c3)[nH][nH]c4=O)C(=O)c2c1 |r|', 'COc1ccc2CN(C[C@]3(NC(=O)NC3=O)C#Cc3ccc(cc3)-c3nc(ccc3O)-c3cnn(C)c3)C(=O)c2c1 |r|', 'COc1ccc2CN(C[C@]3(NC(=O)NC3=O)C#Cc3ccc(cc3)-c3nc(ccc3O)-c3ccncc3)C(=O)c2c1 |r|']


In [10]:
import os
import time
import numpy as np
import pandas as pd
from vina import Vina
from rdkit import Chem
from rdkit.Chem import AllChem
from meeko import MoleculePreparation

In [11]:
# -------------------------
# Paths
# -------------------------
output_dir = "/Users/stuytschaevers/Documents/Yale/spring2026/comp_proteomics/final_project/docking_output_v1/top5_known"
os.makedirs(output_dir, exist_ok=True)

protein_pdbqt = "/Users/stuytschaevers/Documents/Yale/spring2026/comp_proteomics/final_project/pdb_manipulation/protein_chainA_B_fixed.pdbqt"
json_path = "/Users/stuytschaevers/Documents/Yale/spring2026/comp_proteomics/final_project/docking_input/ad4_types.json"


In [12]:
def smiles_to_pdbqt(smiles, output_pdbqt, atom_params):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False

    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDG())
    AllChem.UFFOptimizeMolecule(mol)

    prep = MoleculePreparation(load_atom_params=atom_params)
    prep.prepare(mol)

    with open(output_pdbqt, "w") as f:
        f.write(prep.write_pdbqt_string())

    return True

In [13]:
# Initialize Vina
v = Vina(sf_name="vina")
v.set_receptor(protein_pdbqt)

v.compute_vina_maps(
    center=[-19.28, 74.56, 33.93],
    box_size=[20, 20, 20]
)

# %%
results = []

Computing Vina grid ... done.


In [14]:
for i, smiles in enumerate(smiles_list):
    start_time = time.time()

    ligand_pdbqt = os.path.join(output_dir, f"ligand_top5_{i}.pdbqt")
    docked_pdbqt = os.path.join(output_dir, f"docked_top5_{i}.pdbqt")

    success = smiles_to_pdbqt(smiles, ligand_pdbqt, json_path)
    if not success:
        print(f"[SKIP] {i}")
        continue

    try:
        v.set_ligand_from_file(ligand_pdbqt)

        v.dock(exhaustiveness=16, n_poses=10)
        v.write_poses(docked_pdbqt, n_poses=5, overwrite=True)

        score = v.energies(n_poses=1)[0][0]
        elapsed = time.time() - start_time

        results.append({
            "rank": i + 1,
            "smiles": smiles,
            "vina_score": score,
            "time_sec": elapsed
        })

        print(f"Top {i+1}: {score:.3f} | {elapsed:.2f} sec")

    except Exception as e:
        elapsed = time.time() - start_time

        results.append({
            "rank": i + 1,
            "smiles": smiles,
            "vina_score": None,
            "time_sec": elapsed
        })

        print(f"[FAIL] Top {i+1}: {e}")

/opt/miniconda3/envs/comp_prot/lib/python3.10/site-packages/meeko/preparation.py:626: DeprecationWarning: MoleculePreparation.write_pdbqt_string() is deprecated in Meeko v0.5. Pass the MoleculeSetup instance to PDBQTWriterLegacy.write_string(). MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)
/opt/miniconda3/envs/comp_prot/lib/python3.10/site-packages/meeko/preparation.py:420: DeprecationWarning: MoleculePreparation.setup is deprecated in Meeko v0.5. MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)


Performing docking (random seed: 1706660644) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -8.049          0          0
   2       -7.692      1.067      1.681
   3       -7.662      3.154      6.047
   4       -7.649      3.385      4.469
   5       -7.629      4.189      5.764
   6       -7.577      3.686      5.569
   7       -7.469      2.804      3.687
   8       -7.369      4.698       8.03
   9       -7.367      2.666      6.443
  10       -7.358      6.709      9.193
Top 1: -8.049 | 7.12 sec
Performing docking (random seed: 1706660644) ... Top 2: -8.784 | 6.38 sec

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |  

In [15]:
results_df = pd.DataFrame(results)

output_csv = os.path.join(output_dir, "top5_known_docking_scores.csv")
results_df.to_csv(output_csv, index=False)

results_df

,rank,smiles,vina_score,time_sec
0,1,COc1ccc2CN(C[C@]3(NC(=O)NC3=O)C#Cc3cncc(c3)-c3...,-8.049,7.121344
1,2,COc1ccc2CN(C[C@]3(NC(=O)NC3=O)C#Cc3cc(F)c4c(c3...,-8.784,6.382211
2,3,COc1ccc2CN(C[C@]3(NC(=O)NC3=O)C#Cc3cc(F)c4c(c3...,-8.308,5.896954
3,4,COc1ccc2CN(C[C@]3(NC(=O)NC3=O)C#Cc3ccc(cc3)-c3...,-7.791,14.872709
4,5,COc1ccc2CN(C[C@]3(NC(=O)NC3=O)C#Cc3ccc(cc3)-c3...,-8.711,15.011633
